# Huấn luyện Mô hình Transformer Seq2Seq Thực tế (Multi-GPU DDP & AMP) trên Kaggle

Notebook này dùng để tiến hành huấn luyện mô hình tóm tắt văn bản tiếng Việt **thực tế** (Full Training) theo cấu hình chi tiết trong file `configs/transformer_summarization.yaml`.

## Các tính năng tự động hóa nổi bật:
1. **Tự động cấu hình môi trường**: Tự quét dữ liệu và code từ Kaggle Input Datasets để liên kết mềm và copy vào thư mục làm việc.
2. **Hỗ trợ Resume Training**: Nếu bạn có checkpoint cũ (`checkpoint.zip` chứa `last.pt`), hệ thống sẽ tự động giải nén và tiếp tục huấn luyện từ epoch bị gián đoạn (không phải train lại từ đầu).
3. **Huấn luyện DDP (Multi-GPU)**: Chạy song song trên 2 GPU Tesla T4 của Kaggle.
4. **Đo đạc bộ nhớ & Thời gian**: Log chi tiết VRAM đỉnh (Allocated & Reserved) của mỗi GPU.
5. **Tự động đóng gói checkpoint**: Sau khi hoàn thành hoặc dừng lại, notebook tự động nén file checkpoint mới nhất thành file zip và cung cấp link tải trực tiếp.

In [1]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q sentencepiece pyyaml torch tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [2]:
import os
import sys
import shutil
import zipfile

# Hàm hiển thị cấu trúc thư mục đầu vào trên Kaggle để chẩn đoán khi gặp lỗi
def print_kaggle_input_tree():
    print("\n=== CẤU TRÚC THƯ MỤC /kaggle/input ===")
    if not os.path.exists("/kaggle/input"):
        print("Thư mục /kaggle/input không tồn tại!")
        return
    for root, dirs, files in os.walk("/kaggle/input"):
        level = root.replace("/kaggle/input", "").count(os.sep)
        indent = " " * 4 * level
        print(f"{indent}{os.path.basename(root) or root}/")
        sub_indent = " " * 4 * (level + 1)
        for f in files[:10]: # giới hạn in tối đa 10 file mỗi folder
            print(f"{sub_indent}{f}")
        if len(files) > 10:
            print(f"{sub_indent}... và {len(files) - 10} file khác")

# 2. Tự động liên kết dữ liệu / sao chép mã nguồn trên Kaggle
if os.path.exists("/kaggle"):
    print("Đang chạy trên Kaggle. Bắt đầu tự động cấu hình môi trường...")
    
    tokenizer_dir = None
    bin_dir = None
    src_dir = None
    configs_dir = None
    
    for root, dirs, files in os.walk("/kaggle/input"):
        # Tìm thư mục chứa file tokenizer
        if "vietnamese_spm.model" in files and not tokenizer_dir:
            tokenizer_dir = root
            print(f"-> Tìm thấy thư mục tokenizer tại: {tokenizer_dir}")
        # Tìm thư mục chứa file train_token_id.jsonl
        if "train_token_id.jsonl" in files and not bin_dir:
            bin_dir = root
            print(f"-> Tìm thấy thư mục dữ liệu (bin) tại: {bin_dir}")
        # Tìm thư mục chứa train.py
        if "train.py" in files and "dataset.py" in files and not src_dir:
            src_dir = root
            print(f"-> Tìm thấy thư mục src tại: {src_dir}")
        # Tìm thư mục chứa file cấu hình yaml
        if "transformer_summarization.yaml" in files and not configs_dir:
            configs_dir = root
            print(f"-> Tìm thấy thư mục configs tại: {configs_dir}")
            
    # Kiểm tra xem có thiếu thư mục nào không
    missing = []
    if not tokenizer_dir: missing.append("vietnamese_spm.model (Tokenizer)")
    if not bin_dir: missing.append("train_token_id.jsonl (Dữ liệu huấn luyện)")
    if not src_dir: missing.append("src/train.py (Mã nguồn)")
    if not configs_dir: missing.append("configs/transformer_summarization.yaml (Cấu hình)")
    
    if missing:
        print("\n" + "="*80)
        print("LỖI CẤU HÌNH: Không tìm thấy một số thành phần quan trọng trên Kaggle Input Datasets:")
        for m in missing:
            print(f"  - {m}")
        print("="*80)
        print_kaggle_input_tree()
        raise FileNotFoundError(f"Không tìm thấy các tài nguyên cần thiết: {', '.join(missing)}")
            
    # Tạo cấu trúc thư mục data/ ở thư mục hiện tại (/kaggle/working/)
    os.makedirs("data", exist_ok=True)
    
    # 2.1 Sao chép trực tiếp file tokenizer (Cực kỳ quan trọng: Tránh lỗi symlink khi nạp SentencePiece trong tiến trình DDP)
    if tokenizer_dir:
        dest_tokenizer_dir = "data/tokenizer"
        if os.path.lexists(dest_tokenizer_dir) or os.path.islink(dest_tokenizer_dir):
            if os.path.islink(dest_tokenizer_dir):
                os.unlink(dest_tokenizer_dir)
            else:
                shutil.rmtree(dest_tokenizer_dir)
        os.makedirs(dest_tokenizer_dir, exist_ok=True)
        
        # Sao chép các file .model và .vocab cần thiết
        model_copied = False
        for item in os.listdir(tokenizer_dir):
            if item.endswith(".model") or item.endswith(".vocab"):
                src_file = os.path.join(tokenizer_dir, item)
                dst_file = os.path.join(dest_tokenizer_dir, item)
                if os.path.exists(dst_file):
                    os.remove(dst_file)
                shutil.copy(src_file, dst_file)
                if item.endswith(".model"):
                    model_copied = True
        if model_copied:
            print("✓ Đã sao chép trực tiếp file tokenizer vào data/tokenizer/ (Đảm bảo an toàn, tránh lỗi symlink).")
        else:
            raise FileNotFoundError("Mặc dù tìm thấy thư mục tokenizer nhưng không thể sao chép tệp tin vietnamese_spm.model!")
        
    # 2.2 Tạo liên kết mềm (symlink) cho bin dữ liệu (dùng symlink vì dung lượng file dữ liệu lớn)
    if bin_dir:
        dest_bin_dir = "data/bin"
        if os.path.lexists(dest_bin_dir) or os.path.islink(dest_bin_dir):
            if os.path.islink(dest_bin_dir):
                os.unlink(dest_bin_dir)
            else:
                shutil.rmtree(dest_bin_dir)
        os.symlink(bin_dir, dest_bin_dir)
        print("✓ Đã tạo liên kết mềm data/bin chỉ tới thư mục dữ liệu.")
        
    # 2.3 Sao chép src và configs nếu chúng nằm trong input dataset và chưa có ở working directory
    if src_dir and not os.path.exists("src"):
        shutil.copytree(src_dir, "src")
        print("✓ Đã sao chép thư mục src vào thư mục làm việc hiện tại.")
    if configs_dir and not os.path.exists("configs"):
        shutil.copytree(configs_dir, "configs")
        print("✓ Đã sao chép thư mục configs vào thư mục làm việc hiện tại.")
        
    # 3. Tự động kiểm tra và chuẩn bị checkpoint cũ để Resume
    os.makedirs("checkpoints/transformer_base", exist_ok=True)
    checkpoint_zip_path = None
    checkpoint_pt_path = None
    
    # Quét tìm checkpoint trong working directory hoặc input datasets
    if os.path.exists("checkpoint.zip"):
        checkpoint_zip_path = "checkpoint.zip"
    elif os.path.exists("checkpoints/transformer_base/last.pt"):
        checkpoint_pt_path = "checkpoints/transformer_base/last.pt"
    else:
        for root, dirs, files in os.walk("/kaggle/input"):
            if "checkpoint.zip" in files:
                checkpoint_zip_path = os.path.join(root, "checkpoint.zip")
                break
            if "last.pt" in files:
                checkpoint_pt_path = os.path.join(root, "last.pt")
                break
                
    if checkpoint_zip_path:
        print(f"-> Phát hiện file checkpoint dạng nén tại: {checkpoint_zip_path}")
        # Giải nén đè trực tiếp
        with zipfile.ZipFile(checkpoint_zip_path, 'r') as zip_ref:
            zip_ref.extractall("checkpoints/transformer_base")
        print("✓ Đã giải nén checkpoint thành công vào checkpoints/transformer_base/last.pt.")
    elif checkpoint_pt_path:
        print(f"-> Phát hiện file checkpoint .pt trực tiếp tại: {checkpoint_pt_path}")
        dest_pt = "checkpoints/transformer_base/last.pt"
        if checkpoint_pt_path != dest_pt:
            if os.path.exists(dest_pt):
                os.remove(dest_pt)
            shutil.copy(checkpoint_pt_path, dest_pt)
        print("✓ Đã sao chép checkpoint vào checkpoints/transformer_base/last.pt.")

# Thêm thư mục vào python path
if os.path.exists("../src"):
    sys.path.append(os.path.abspath(".."))
else:
    sys.path.append(os.path.abspath("."))


Đang chạy trên Kaggle. Bắt đầu tự động cấu hình môi trường...
-> Tìm thấy thư mục tokenizer tại: /kaggle/input/datasets/tranducthinh2006/vietnamese-transformer-summarization-data/data/tokenizer
-> Tìm thấy thư mục configs tại: /kaggle/input/datasets/tranducthinh2006/vietnamese-transformer-summarization-data/data/configs
-> Tìm thấy thư mục src tại: /kaggle/input/datasets/tranducthinh2006/vietnamese-transformer-summarization-data/data/src
-> Tìm thấy thư mục dữ liệu (bin) tại: /kaggle/input/datasets/tranducthinh2006/vietnamese-transformer-summarization-data/data/bin
✓ Đã sao chép trực tiếp file tokenizer vào data/tokenizer/ (Đảm bảo an toàn, tránh lỗi symlink).
✓ Đã tạo liên kết mềm data/bin chỉ tới thư mục dữ liệu.
✓ Đã sao chép thư mục src vào thư mục làm việc hiện tại.
✓ Đã sao chép thư mục configs vào thư mục làm việc hiện tại.
-> Phát hiện file checkpoint .pt trực tiếp tại: /kaggle/input/datasets/tranducthinh2006/my-checkpoint-dataset/last.pt
✓ Đã sao chép checkpoint vào checkpoint

## Kiểm tra Phần cứng GPU

In [3]:
import torch
print("=== THÔNG TIN PHẦN CỨNG GPU ===")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
print(f"Số lượng GPU khả dụng: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

=== THÔNG TIN PHẦN CỨNG GPU ===
CUDA khả dụng: True
Số lượng GPU khả dụng: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 1. Chạy Đo lường VRAM và Tốc độ (Benchmark)

Chạy cell dưới đây để tự động đo đạc mức độ chiếm dụng VRAM và tốc độ huấn luyện ước tính trên GPU T4 đối với các kích thước batch khác nhau (8, 16, 32, 64, 128).

In [4]:
!python src/benchmark.py

Thiết bị đo lường: cuda
Đang nạp dataset huấn luyện...
Tổng số mẫu trong tập huấn luyện: 193841

--- Đang kiểm tra Batch Size: 8 ---
/kaggle/working/src/benchmark.py:105: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=t_cfg.get("use_amp", True) and device.type == "cuda"):
/kaggle/working/src/benchmark.py:133: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=t_cfg.get("use_amp", True) and device.type == "cuda"):
✓ Hoàn thành benchmark cho Batch Size: 8

--- Đang kiểm tra Batch Size: 16 ---
✓ Hoàn thành benchmark cho Batch Size: 16

--- Đang kiểm tra Batch Size: 32 ---
✓ Hoàn thành benchmark cho Batch Size: 32

--- Đang kiểm tra Batch Size: 64 ---
✗ LỖI OOM: Hết bộ nhớ GPU tại Batch Size: 64

--- Đang kiểm tra Batch Size: 128 ---
✗ LỖI OOM: Hết bộ nhớ GPU tại Batc

## 2. Điền Batch Size Tối ưu Bạn Chọn

Sau khi xem bảng so sánh ở cell trên, hãy **điền batch size bạn muốn chọn vào biến `SELECTED_BATCH_SIZE`** dưới đây, sau đó chạy cell này. 

Hệ thống sẽ tự động cập nhật file cấu hình YAML (`configs/transformer_summarization.yaml`) và tự căn chỉnh `gradient_accumulation_steps` để giữ nguyên kích thước batch hiệu dụng là `64` cho bạn.

In [5]:
# =========================================================
# HÃY ĐIỀN BATCH SIZE TỐT NHẤT CHẠY THÀNH CÔNG VÀO ĐÂY:
SELECTED_BATCH_SIZE = 32
# =========================================================

import os
import yaml

# Tự động tính toán gradient_accumulation_steps (Effective Batch = SELECTED_BATCH_SIZE * grad_accum * 2 GPU = 64)
grad_accum = 64 // (SELECTED_BATCH_SIZE * 2)
grad_accum = max(1, grad_accum)

yaml_path = "configs/transformer_summarization.yaml"
if os.path.exists(yaml_path):
    with open(yaml_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    config["training"]["batch_size_per_device"] = SELECTED_BATCH_SIZE
    config["training"]["gradient_accumulation_steps"] = grad_accum
    config["training"]["epochs"] = 20
    
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, allow_unicode=True, default_flow_style=False)
        
    print(f"✓ Đã cập nhật thành công file cấu hình YAML!")
    print(f"  - batch_size_per_device: {SELECTED_BATCH_SIZE}")
    print(f"  - gradient_accumulation_steps: {grad_accum}")
    print(f"  - Kích thước batch hiệu dụng (Effective Batch Size): {SELECTED_BATCH_SIZE * grad_accum * 2}")
else:
    print("LỖI: Không tìm thấy file cấu hình configs/transformer_summarization.yaml")

✓ Đã cập nhật thành công file cấu hình YAML!
  - batch_size_per_device: 32
  - gradient_accumulation_steps: 1
  - Kích thước batch hiệu dụng (Effective Batch Size): 64


## Kích hoạt huấn luyện Multi-GPU DDP (Hỗ trợ tự động Resume)

In [6]:
import subprocess
import os

# Khởi dựng lệnh chạy torchrun DDP
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "src/train.py",
    "--config", "configs/transformer_summarization.yaml"
]

# Tự động phát hiện checkpoint để kích hoạt chế độ Resume
checkpoint_file = "checkpoints/transformer_base/last.pt"
if os.path.exists(checkpoint_file):
    print("-> Phát hiện checkpoint cũ. Tự động kích hoạt chế độ HUẤN LUYỆN TIẾP TỤC (Resume)...")
    cmd.extend(["--resume_from", checkpoint_file])
else:
    print("-> Không tìm thấy checkpoint cũ. Bắt đầu HUẤN LUYỆN MỚI TỪ ĐẦU...")

print(f"Lệnh thực thi: {' '.join(cmd)}")

# Thực thi tiến trình và in log thời gian thực (Real-time stdout log)
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

try:
    for line in process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\n[CẢNH BÁO] Huấn luyện bị dừng bởi người dùng (KeyboardInterrupt).")
    process.terminate()

process.wait()
if process.returncode != 0:
    print(f"\n[LỖI] Tiến trình huấn luyện kết thúc với mã lỗi: {process.returncode}")
else:
    print("\n✓ Huấn luyện hoàn thành thành công!")

-> Phát hiện checkpoint cũ. Tự động kích hoạt chế độ HUẤN LUYỆN TIẾP TỤC (Resume)...
Lệnh thực thi: torchrun --nproc_per_node=2 src/train.py --config configs/transformer_summarization.yaml --resume_from checkpoints/transformer_base/last.pt
W0608 16:30:50.666000 60 torch/distributed/run.py:852] 
W0608 16:30:50.666000 60 torch/distributed/run.py:852] *****************************************
W0608 16:30:50.666000 60 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0608 16:30:50.666000 60 torch/distributed/run.py:852] *****************************************
[2026-06-08 16:30:53] INFO [TrainPipeline:85] Khởi chạy tiến trình. Rank: 0, World Size: 2, Device: cuda:0, DDP: True
[2026-06-08 16:30:53] INFO [TrainPipeline:101] Đang tải Tokenizer từ: data/tokenizer/vietnamese_spm.model
[2026-06-08 16:30:53]

## Nén và tải về checkpoint mới nhất

Sau khi phiên huấn luyện kết thúc hoặc bị dừng lại, hãy chạy ô code dưới đây để đóng gói checkpoint mới nhất thành file zip và click link hiển thị bên dưới để tải trực tiếp về máy tính.

In [7]:
import os
import shutil
from IPython.display import FileLink

checkpoint_dir = "checkpoints/transformer_base"
if os.path.exists(checkpoint_dir) and os.listdir(checkpoint_dir):
    print("-> Đang đóng gói toàn bộ các checkpoint trong thư mục...")
    # Nén toàn bộ thư mục checkpoints/transformer_base thành checkpoint.zip
    shutil.make_archive("checkpoint", "zip", root_dir=checkpoint_dir)
    print("✓ Đã tạo thành công file: checkpoint.zip (chứa cả last.pt và best_val_loss.pt)")
    # Hiển thị link tải trực tiếp
    display(FileLink("checkpoint.zip"))
else:
    print("Không tìm thấy thư mục checkpoint để đóng gói.")

-> Đang đóng gói toàn bộ các checkpoint trong thư mục...
✓ Đã tạo thành công file: checkpoint.zip (chứa cả last.pt và best_val_loss.pt)


/kaggle/working/checkpoint.zip